## Imports

In [1]:
import os
import time
import glob
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [3]:
## Connection
connection = presto.connect(
        host= 'processing-2.processing.data.production.internal',
    # 'presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

## Datasets & Parameter

In [5]:
segment_query = f"""
    
    with base_segment as (
    
        select 
            customer_id,
            segment,
            category appCategory,
            use_case useCase
        from 
            hive.experiments_internal.customer_segment_appcategory_usecase_amj2025
        ),
    
    
        funnel as (
        
        select
            date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
            customer_id,
            order_id,
            modified_order_status,
            customer_set_pickup_ride_started_distance_meters,
            amount_breakup_bid_delta_total,
            bid_type,
            drop_location_hex_8
            
        from 
            orders.order_logs_fact
        where 
            yyyymmdd >= '20250217'  
            and yyyymmdd <= '20250316'
            and city_name = 'Bangalore'
            and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
            and channel_host in ('android','ios')
        )
        
        select
            base_segment.segment,
            -- base_segment.appCategory,
            -- base_segment.useCase,
            count(distinct funnel.customer_id) gross_customers,
            count(distinct order_id) gross_orders,
            count(distinct case when modified_order_status = 'dropped' then funnel.customer_id end) net_customers,
            count(distinct case when modified_order_status = 'dropped' then order_id end) net_orders,
            count(distinct case when modified_order_status = 'COBRM' then order_id end) cobrm_orders,
            count(distinct case when modified_order_status = 'COBRA' then order_id end) cobra_orders,
            count(distinct case when modified_order_status = 'OCARA' then order_id end) ocara_orders,
            count(distinct drop_location_hex_8) unique_dropHex8,
            count(distinct case when modified_order_status = 'dropped' and customer_set_pickup_ride_started_distance_meters <= 30.00 then order_id end) hap_orders,
            count(distinct case when amount_breakup_bid_delta_total = 0 then order_id end) zero_bid_gross,
            count(distinct case when amount_breakup_bid_delta_total < 0 then order_id end) negative_bid_gross,
            count(distinct case when amount_breakup_bid_delta_total > 0 then order_id end) positive_bid_gross,
            count(distinct case when modified_order_status = 'dropped' and amount_breakup_bid_delta_total = 0 then order_id end) zero_bid_net,
            count(distinct case when modified_order_status = 'dropped' and amount_breakup_bid_delta_total < 0 then order_id end) negative_bid_net,
            count(distinct case when modified_order_status = 'dropped' and amount_breakup_bid_delta_total > 0 then order_id end) positive_bid_net
            
        from 
            base_segment
        left join 
            funnel
            on base_segment.customer_id = funnel.customer_id
            -- and funnel.week_start_date = '20250310'
            
        group by 1
        order by 1

"""

df_segment_query_v1 = pd.read_sql(segment_query, connection)
df_segment_query_v1.head()

,segment,gross_customers,gross_orders,net_customers,net_orders,cobrm_orders,cobra_orders,ocara_orders,unique_dropHex8,hap_orders,zero_bid_gross,negative_bid_gross,positive_bid_gross,zero_bid_net,negative_bid_net,positive_bid_net
0,1-DAILY,108436,4529786,108307,2834222,28147,669144,759535,2461,2251873,2161538,644461,517442,1424899,416874,286801
1,2-WEEKLY,296366,4604073,294469,2819743,33846,709472,772900,2760,2153066,2430999,492294,452809,1570917,298908,241626
2,3-MONTHLY,1684397,8611910,1531599,5357798,73915,1316809,1378249,3093,3961379,4486045,548170,728502,2924472,335774,387243
3,4-OTHER,768154,2723132,701732,1772103,26625,362796,402323,2943,1270979,1335106,118479,211086,906476,76224,117162
4,5-INACTIVE,85110,431923,77426,275109,3423,66312,64703,2252,206650,204155,31836,30760,135884,20461,16258


In [6]:
df_segment_query_v1

,segment,gross_customers,gross_orders,net_customers,net_orders,cobrm_orders,cobra_orders,ocara_orders,unique_dropHex8,hap_orders,zero_bid_gross,negative_bid_gross,positive_bid_gross,zero_bid_net,negative_bid_net,positive_bid_net
0,1-DAILY,108436,4529786,108307,2834222,28147,669144,759535,2461,2251873,2161538,644461,517442,1424899,416874,286801
1,2-WEEKLY,296366,4604073,294469,2819743,33846,709472,772900,2760,2153066,2430999,492294,452809,1570917,298908,241626
2,3-MONTHLY,1684397,8611910,1531599,5357798,73915,1316809,1378249,3093,3961379,4486045,548170,728502,2924472,335774,387243
3,4-OTHER,768154,2723132,701732,1772103,26625,362796,402323,2943,1270979,1335106,118479,211086,906476,76224,117162
4,5-INACTIVE,85110,431923,77426,275109,3423,66312,64703,2252,206650,204155,31836,30760,135884,20461,16258


In [23]:
df_segment_query_v1.to_clipboard(index=False)

In [8]:
segment_query = f"""
    
    with base_segment as (
    
        select 
            customer_id,
            segment,
            category appCategory,
            use_case useCase
        from 
            hive.experiments_internal.customer_segment_appcategory_usecase_amj2025
        ),
        
        ao_rr_seesion as (
        
        select 
            week_start_date,
            customerid as customer_id,
            cast(ao_session as int) as ao_session,
            cast(fe_session as int) as fe_session,
            cast(rr_session as int) as rr_session
        from 
            hive.experiments_internal.customer_funnel_last90days_20250317
        where 
            week_start_date >= '20250217'
            and week_start_date <= '20250310'
        )
        
        
        select
            base_segment.segment,
            -- base_segment.appCategory,
            -- base_segment.useCase,
            sum(ao_session) ao_session,
            sum(fe_session) fe_session,
            sum(rr_session) rr_session
            
        from 
            base_segment
        left join 
            ao_rr_seesion
            on base_segment.customer_id = ao_rr_seesion.customer_id
            -- and ao_rr_seesion.week_start_date = '20250310'
            
        group by 1
        order by 1

"""

df_segment_query_v2 = pd.read_sql(segment_query, connection)
df_segment_query_v2.head()

,segment,ao_session,fe_session,rr_session
0,1-DAILY,5519284,4133553,3665006
1,2-WEEKLY,6312328,4786191,3805101
2,3-MONTHLY,13967458,10906978,7222324
3,4-OTHER,5093346,3976431,2274994
4,5-INACTIVE,303552,272022,0


In [9]:
df_segment_query_v2

,segment,ao_session,fe_session,rr_session
0,1-DAILY,5519284,4133553,3665006
1,2-WEEKLY,6312328,4786191,3805101
2,3-MONTHLY,13967458,10906978,7222324
3,4-OTHER,5093346,3976431,2274994
4,5-INACTIVE,303552,272022,0


In [24]:
df_segment_query_v2.to_clipboard(index=False)

In [11]:
segment_usecase_query = f"""
    
    with base_segment as (
    
        select 
            customer_id,
            segment,
            category appCategory,
            use_case useCase
        from 
            hive.experiments_internal.customer_segment_appcategory_usecase_amj2025
        ),
    
    
        funnel as (
        
        select
            date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
            customer_id,
            order_id,
            modified_order_status,
            customer_set_pickup_ride_started_distance_meters,
            amount_breakup_bid_delta_total,
            bid_type,
            drop_location_hex_8
            
        from 
            orders.order_logs_fact
        where 
            yyyymmdd >= '20250217'  
            and yyyymmdd <= '20250316'
            and city_name = 'Bangalore'
            and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
            and channel_host in ('android','ios')
        )
        
        
        select
            base_segment.segment,
            -- base_segment.appCategory,
            base_segment.useCase,
            count(distinct funnel.customer_id) gross_customers,
            count(distinct order_id) gross_orders,
            count(distinct case when modified_order_status = 'dropped' then funnel.customer_id end) net_customers,
            count(distinct case when modified_order_status = 'dropped' then order_id end) net_orders,
            count(distinct case when modified_order_status = 'COBRM' then order_id end) cobrm_orders,
            count(distinct case when modified_order_status = 'COBRA' then order_id end) cobra_orders,
            count(distinct case when modified_order_status = 'OCARA' then order_id end) ocara_orders,
            count(distinct drop_location_hex_8) unique_dropHex8,
            count(distinct case when modified_order_status = 'dropped' and customer_set_pickup_ride_started_distance_meters <= 30.00 then order_id end) hap_orders,
            count(distinct case when amount_breakup_bid_delta_total = 0 then order_id end) zero_bid_gross,
            count(distinct case when amount_breakup_bid_delta_total < 0 then order_id end) negative_bid_gross,
            count(distinct case when amount_breakup_bid_delta_total > 0 then order_id end) positive_bid_gross,
            count(distinct case when modified_order_status = 'dropped' and amount_breakup_bid_delta_total = 0 then order_id end) zero_bid_net,
            count(distinct case when modified_order_status = 'dropped' and amount_breakup_bid_delta_total < 0 then order_id end) negative_bid_net,
            count(distinct case when modified_order_status = 'dropped' and amount_breakup_bid_delta_total > 0 then order_id end) positive_bid_net            
        from 
            base_segment
        left join 
            funnel
            on base_segment.customer_id = funnel.customer_id
            -- and funnel.week_start_date = '20250310'
            
        group by 1,2
        order by 1,2

"""

df_segment_usecase_query_v1 = pd.read_sql(segment_usecase_query, connection)
df_segment_usecase_query_v1.head()

,segment,useCase,gross_customers,gross_orders,net_customers,net_orders,cobrm_orders,cobra_orders,ocara_orders,unique_dropHex8,hap_orders,zero_bid_gross,negative_bid_gross,positive_bid_gross,zero_bid_net,negative_bid_net,positive_bid_net
0,1-DAILY,a. office,87015,3735468,86963,2327455,23259,556212,631474,2384,1842115,1770343,531697,454943,1161531,342142,253641
1,1-DAILY,b. transit,10459,399892,10445,257640,2561,56117,64120,1787,207083,193298,60454,30297,131527,40231,16183
2,1-DAILY,c. other,10962,394426,10899,249127,2327,56815,63941,1706,202675,197897,52310,32202,131841,34501,16977
3,2-WEEKLY,a. office,208986,3431393,208037,2092400,25492,536271,574268,2621,1586010,1782361,377859,367378,1144956,228298,198386
4,2-WEEKLY,b. transit,41570,569220,41266,359951,4192,83945,92477,2231,277183,307894,57781,38720,206722,36095,19931


In [12]:
df_segment_usecase_query_v1

,segment,useCase,gross_customers,gross_orders,net_customers,net_orders,cobrm_orders,cobra_orders,ocara_orders,unique_dropHex8,hap_orders,zero_bid_gross,negative_bid_gross,positive_bid_gross,zero_bid_net,negative_bid_net,positive_bid_net
0,1-DAILY,a. office,87015,3735468,86963,2327455,23259,556212,631474,2384,1842115,1770343,531697,454943,1161531,342142,253641
1,1-DAILY,b. transit,10459,399892,10445,257640,2561,56117,64120,1787,207083,193298,60454,30297,131527,40231,16183
2,1-DAILY,c. other,10962,394426,10899,249127,2327,56815,63941,1706,202675,197897,52310,32202,131841,34501,16977
3,2-WEEKLY,a. office,208986,3431393,208037,2092400,25492,536271,574268,2621,1586010,1782361,377859,367378,1144956,228298,198386
4,2-WEEKLY,b. transit,41570,569220,41266,359951,4192,83945,92477,2231,277183,307894,57781,38720,206722,36095,19931
5,2-WEEKLY,c. other,45810,603460,45166,367392,4162,89256,106155,2196,289873,340744,56654,46711,219239,34515,23309
6,3-MONTHLY,a. office,706130,4651267,669183,2906039,38771,726063,719879,2800,2130914,2366587,332552,441132,1542601,203199,242158
7,3-MONTHLY,b. transit,314694,1461445,293232,942732,12361,213447,223299,2689,687056,768855,82387,99339,524738,52926,52698
8,3-MONTHLY,c. other,663573,2499198,569184,1509027,22783,377299,435071,2871,1143409,1350603,133231,188031,857133,79649,92387
9,4-OTHER,a. office,157657,946038,155912,622246,9111,133080,129354,2345,442122,451154,48945,85017,308743,31492,49407


In [25]:
df_segment_usecase_query_v1.to_clipboard(index=False)

In [13]:
segment_usecase_query = f"""
    
    with base_segment as (
    
        select 
            customer_id,
            segment,
            category appCategory,
            use_case useCase
        from 
            hive.experiments_internal.customer_segment_appcategory_usecase_amj2025
        ),
        
        ao_rr_seesion as (
        
        select 
            week_start_date,
            customerid as customer_id,
            cast(ao_session as int) as ao_session,
            cast(fe_session as int) as fe_session,
            cast(rr_session as int) as rr_session
        from 
            hive.experiments_internal.customer_funnel_last90days_20250317
        where 
            week_start_date >= '20250217'
            and week_start_date <= '20250310'
        )
        
        
        select
            base_segment.segment,
            -- base_segment.appCategory,
            base_segment.useCase,
            sum(ao_session) ao_session,
            sum(fe_session) fe_session,
            sum(rr_session) rr_session
            
        from 
            base_segment
        left join 
            ao_rr_seesion
            on base_segment.customer_id = ao_rr_seesion.customer_id
            -- and ao_rr_seesion.week_start_date = '20250310'
            
        group by 1,2
        order by 1,2

"""

df_segment_usecase_query_v2 = pd.read_sql(segment_usecase_query, connection)
df_segment_usecase_query_v2.head()

,segment,useCase,ao_session,fe_session,rr_session
0,1-DAILY,a. office,4581081,3397011,3021321
1,1-DAILY,b. transit,477470,373349,325367
2,1-DAILY,c. other,460733,363193,318318
3,2-WEEKLY,a. office,4705125,3529949,2833211
4,2-WEEKLY,b. transit,788059,616575,473381


In [14]:
df_segment_usecase_query_v2

,segment,useCase,ao_session,fe_session,rr_session
0,1-DAILY,a. office,4581081,3397011,3021321
1,1-DAILY,b. transit,477470,373349,325367
2,1-DAILY,c. other,460733,363193,318318
3,2-WEEKLY,a. office,4705125,3529949,2833211
4,2-WEEKLY,b. transit,788059,616575,473381
5,2-WEEKLY,c. other,819144,639667,498509
6,3-MONTHLY,a. office,7216680,5545021,3873522
7,3-MONTHLY,b. transit,2455760,1949696,1233180
8,3-MONTHLY,c. other,4295018,3412261,2115622
9,4-OTHER,a. office,1556383,1187187,768201


In [26]:
df_segment_usecase_query_v2.to_clipboard(index=False)

In [17]:
segment_appcat_usecase_query = f"""
    
    with base_segment as (
    
        select 
            customer_id,
            segment,
            category appCategory,
            use_case useCase
        from 
            hive.experiments_internal.customer_segment_appcategory_usecase_amj2025
        ),
    
    
        funnel as (
        
        select
            date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
            customer_id,
            order_id,
            modified_order_status,
            customer_set_pickup_ride_started_distance_meters,
            amount_breakup_bid_delta_total,
            bid_type,
            drop_location_hex_8
            
        from 
            orders.order_logs_fact
        where 
            yyyymmdd >= '20250217'  
            and yyyymmdd <= '20250316'
            and city_name = 'Bangalore'
            and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
            and channel_host in ('android','ios')
        )
        
        select
            base_segment.segment,
            base_segment.appCategory,
            base_segment.useCase,
            count(distinct funnel.customer_id) gross_customers,
            count(distinct order_id) gross_orders,
            count(distinct case when modified_order_status = 'dropped' then funnel.customer_id end) net_customers,
            count(distinct case when modified_order_status = 'dropped' then order_id end) net_orders,
            count(distinct case when modified_order_status = 'COBRM' then order_id end) cobrm_orders,
            count(distinct case when modified_order_status = 'COBRA' then order_id end) cobra_orders,
            count(distinct case when modified_order_status = 'OCARA' then order_id end) ocara_orders,
            count(distinct drop_location_hex_8) unique_dropHex8,
            count(distinct case when modified_order_status = 'dropped' and customer_set_pickup_ride_started_distance_meters <= 30.00 then order_id end) hap_orders,
            count(distinct case when amount_breakup_bid_delta_total = 0 then order_id end) zero_bid_gross,
            count(distinct case when amount_breakup_bid_delta_total < 0 then order_id end) negative_bid_gross,
            count(distinct case when amount_breakup_bid_delta_total > 0 then order_id end) positive_bid_gross,
            count(distinct case when modified_order_status = 'dropped' and amount_breakup_bid_delta_total = 0 then order_id end) zero_bid_net,
            count(distinct case when modified_order_status = 'dropped' and amount_breakup_bid_delta_total < 0 then order_id end) negative_bid_net,
            count(distinct case when modified_order_status = 'dropped' and amount_breakup_bid_delta_total > 0 then order_id end) positive_bid_net
            
        from 
            base_segment
        left join 
            funnel
            on base_segment.customer_id = funnel.customer_id
            -- and funnel.week_start_date = '20250310'
            
        group by 1,2,3
        order by 1,2,3

"""

df_segment_appcat_query_v1 = pd.read_sql(segment_appcat_usecase_query, connection)
df_segment_appcat_query_v1.head()

,segment,appCategory,useCase,gross_customers,gross_orders,net_customers,net_orders,cobrm_orders,cobra_orders,ocara_orders,unique_dropHex8,hap_orders,zero_bid_gross,negative_bid_gross,positive_bid_gross,zero_bid_net,negative_bid_net,positive_bid_net
0,1-DAILY,a. RHA - NQC,a. office,3869,161745,3853,103400,1245,23885,24417,1677,81691,73250,18816,10156,48813,12535,5370
1,1-DAILY,a. RHA - NQC,b. transit,954,36001,951,23263,290,5139,5332,1107,18463,16274,4635,2162,11033,3089,977
2,1-DAILY,a. RHA - NQC,c. other,985,36303,965,22334,307,5792,6056,1110,18363,17333,3952,2084,11235,2653,863
3,1-DAILY,a. RHA - QC,a. office,47635,2038893,47617,1264317,13473,318892,334864,2151,1004623,932620,293323,243569,611013,188562,133767
4,1-DAILY,a. RHA - QC,b. transit,5811,221703,5808,142142,1511,32037,35375,1614,114781,106341,34811,17468,72356,23230,9369


In [18]:
df_segment_appcat_query_v1

,segment,appCategory,useCase,gross_customers,gross_orders,net_customers,net_orders,cobrm_orders,cobra_orders,ocara_orders,unique_dropHex8,hap_orders,zero_bid_gross,negative_bid_gross,positive_bid_gross,zero_bid_net,negative_bid_net,positive_bid_net
0,1-DAILY,a. RHA - NQC,a. office,3869,161745,3853,103400,1245,23885,24417,1677,81691,73250,18816,10156,48813,12535,5370
1,1-DAILY,a. RHA - NQC,b. transit,954,36001,951,23263,290,5139,5332,1107,18463,16274,4635,2162,11033,3089,977
2,1-DAILY,a. RHA - NQC,c. other,985,36303,965,22334,307,5792,6056,1110,18363,17333,3952,2084,11235,2653,863
3,1-DAILY,a. RHA - QC,a. office,47635,2038893,47617,1264317,13473,318892,334864,2151,1004623,932620,293323,243569,611013,188562,133767
4,1-DAILY,a. RHA - QC,b. transit,5811,221703,5808,142142,1511,32037,35375,1614,114781,106341,34811,17468,72356,23230,9369
5,1-DAILY,a. RHA - QC,c. other,6108,218881,6076,137031,1504,32080,35511,1529,111523,109997,31194,18568,73000,20444,9588
6,1-DAILY,b. NRHA - NQC,a. office,3493,147813,3487,95603,971,23326,22088,1575,74866,63622,15766,9844,43036,10625,5339
7,1-DAILY,b. NRHA - NQC,b. transit,774,29379,771,19362,162,4317,4252,1024,15422,13397,2936,1557,9260,2049,807
8,1-DAILY,b. NRHA - NQC,c. other,876,30505,866,19828,177,4684,4462,931,16094,14267,3260,1588,9874,2116,830
9,1-DAILY,b. NRHA - QC,a. office,5804,245943,5800,160308,1455,38488,37154,1546,127345,117511,25065,24186,81210,17151,13685


In [27]:
df_segment_appcat_query_v1.to_clipboard(index=False)

In [20]:
segment_appcat_usecase_query = f"""
    
    with base_segment as (
    
        select 
            customer_id,
            segment,
            category appCategory,
            use_case useCase
        from 
            hive.experiments_internal.customer_segment_appcategory_usecase_amj2025
        ),
            
        ao_rr_seesion as (
        
        select 
            week_start_date,
            customerid as customer_id,
            cast(ao_session as int) as ao_session,
            cast(fe_session as int) as fe_session,
            cast(rr_session as int) as rr_session
        from 
            hive.experiments_internal.customer_funnel_last90days_20250317
        where 
            week_start_date >= '20250217'
            and week_start_date <= '20250310'
        )
        
        
        select
            base_segment.segment,
            base_segment.appCategory,
            base_segment.useCase,
            sum(ao_session) ao_session,
            sum(fe_session) fe_session,
            sum(rr_session) rr_session
            
        from 
            base_segment
        left join 
            ao_rr_seesion
            on base_segment.customer_id = ao_rr_seesion.customer_id
            -- and ao_rr_seesion.week_start_date = '20250310'
            
        group by 1,2,3
        order by 1,2,3

"""

df_segment_appcat_query_v2 = pd.read_sql(segment_appcat_usecase_query, connection)
df_segment_appcat_query_v2.head()

,segment,appCategory,useCase,ao_session,fe_session,rr_session
0,1-DAILY,a. RHA - NQC,a. office,201645,157771,130770
1,1-DAILY,a. RHA - NQC,b. transit,42639,34485,28940
2,1-DAILY,a. RHA - NQC,c. other,41981,34221,28549
3,1-DAILY,a. RHA - QC,a. office,2235388,1819809,1623070
4,1-DAILY,a. RHA - QC,b. transit,248956,205342,179045


In [21]:
df_segment_appcat_query_v2

,segment,appCategory,useCase,ao_session,fe_session,rr_session
0,1-DAILY,a. RHA - NQC,a. office,201645,157771,130770
1,1-DAILY,a. RHA - NQC,b. transit,42639,34485,28940
2,1-DAILY,a. RHA - NQC,c. other,41981,34221,28549
3,1-DAILY,a. RHA - QC,a. office,2235388,1819809,1623070
4,1-DAILY,a. RHA - QC,b. transit,248956,205342,179045
5,1-DAILY,a. RHA - QC,c. other,238006,198706,174902
6,1-DAILY,b. NRHA - NQC,a. office,176592,136040,117424
7,1-DAILY,b. NRHA - NQC,b. transit,34383,27091,23401
8,1-DAILY,b. NRHA - NQC,c. other,34504,27474,24023
9,1-DAILY,b. NRHA - QC,a. office,271967,216462,194026


In [28]:
df_segment_appcat_query_v2.to_clipboard(index=False)